# AIFM Hedge Fund

This notebook presents a hedge fund risk-monitoring workflow for a simulated AIF portfolio. The analysis uses fund-level Risk Management Policy parameters, portfolio and market-data inputs from the data layer, and selected analytics for market risk, liquidity risk, leverage, stress testing, attribution, pre-trade controls, sustainability indicators, and Annex IV-style reporting outputs.

The regulatory context is AIFMD-style risk governance. The focus is the monitoring workflow and the interpretation of the resulting risk metrics.

> **Output gallery:** All tables and plots generated by this notebook are saved in the [fig/AIFM_HedgeFund](../../fig/AIFM_HedgeFund) folder. Readers who prefer to review the generated outputs directly can browse that folder without running the notebook.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Initialise the database if needed and create the database engine
from src.setup_db import run as setup_db
setup_db()
from src.data.database import get_engine
ENGINE = get_engine()

# Initialise the mock Bloomberg API
from src.data.mock_bloomberg import MockBloomberg as Bloomberg
BBG = Bloomberg()

# Helper functions for risk calculations and output formatting
import src.risk.risk_utils as rk
import src.ui.print_html_utils as phtml

## 1. Fund Setup and Risk Policy

### 1.1 Fund Example

The fund profile below sets the operating context for the risk workflow. It defines the strategy, fund type, base currency, reporting setup, and monitoring framework used by the calculations that follow.

<small>


In [ ]:
# Display fund overview banner — fund identity and risk methodology framework
FUND_ID = 'AIFM_HedgeFund'
phtml.display_fund_overview_banner(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="01",
)

> Note: Fund characteristics, risk limits, methodologies, and reporting parameters are simulated. They are used to show how a fund-level risk framework can be represented in a structured workflow.

---

### 1.2 Risk Management Policy Parameters

The fund's risk parameters are sourced from the Risk Management Policy configuration and used throughout the notebook for measurement, monitoring, and limit checks.


In [ ]:
# Display Risk Management Policy parameters from fund reference data
phtml.display_fund_rmp_parameters(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="02",
)

The RMP is loaded as `rmp` and passed to the relevant risk functions. This keeps fund-specific parameters outside the calculation code.

In [ ]:
# Load RMP parameters for the fund
from src.data.reference_data import load_rmp
rmp = load_rmp(FUND_ID)

### 1.3 Implementation Context

The analysis is performed as of a fixed valuation date, consistent with the point-in-time design used across the fund workflows.


In [ ]:
# Fixed valuation date for all computations in this notebook
from src.config import VALUATION_DATE
VALUATION_DATE

Portfolio positions, fund characteristics, counterparty records, reference data, and market data are retrieved from the SQLite data layer. Market data is enriched through the simulated Bloomberg workflow before being passed to the risk analytics modules.

For a fuller explanation of the data workflow, see the [Data Layer Workflow](../data_workflows/01_data_layer_workflow.ipynb).

In [ ]:
# Enrich fund position data with market-data inputs
from src.data.enrichment import get_risk_ready_df
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)

# Display the first rows of the enriched risk DataFrame
risk_df.head(2)

From this point onward, the notebook uses helper functions to render summary tables and HTML views. Data aggregation, calculations, and reporting logic remain inside the package modules, so the notebook stays focused on review and interpretation.

In [ ]:
# Query positions from SQLite
from src.data.database import query_positions
positions = query_positions(ENGINE, FUND_ID, VALUATION_DATE)
NAV = risk_df['market_value_eur'].sum()

phtml.display_fund_summary(FUND_ID, VALUATION_DATE, positions, risk_df, NAV, export_id="03")

In [ ]:
phtml.display_top_positions(risk_df, valuation_date=VALUATION_DATE, n_top=10, fund_id=FUND_ID, export_id="04")

In [ ]:
phtml.display_asset_class_breakdown(risk_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="05")

## 2. VaR and Expected Shortfall

Market risk is monitored through Historical VaR and Expected Shortfall using the methodology defined in the RMP. The confidence level, holding period and distribution assumptions are applied to the portfolio snapshot.

In [ ]:
# Load VaR parameters from the RMP
var_confidence = rmp['var_framework']['confidence_level']
var_degrees_of_freedom = rmp['var_framework']['parametric_degrees_of_freedom']
var_holding_period = rmp['var_framework']['holding_period_days']
var_lookback = rmp['var_framework']['lookback_period_days']

print(var_confidence, var_degrees_of_freedom, var_holding_period, var_lookback)

In [ ]:
# 1-day fixed-position VaR + ES computation (1 day and scaled)
from src.pipeline.fixed_position_var import compute_fixed_position_var_1day

var_result = compute_fixed_position_var_1day(
    engine=ENGINE,
    fund_id=FUND_ID,
    valuation_date=VALUATION_DATE,
    confidence=var_confidence,
    df=var_degrees_of_freedom,
    horizon=var_holding_period,
)

# Display VaR and ES (extracts nav, horizon, and valuation_date from var_result)
phtml.display_var_es(var_result, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="06")

### 2.1 Historical VaR and Expected Shortfall


### 2.2 Liquidity-Adjusted VaR

Liquidity-Adjusted VaR adds an estimated liquidation cost to market VaR under stressed trading conditions:

$$\text{LVaR} = \text{VaR} + \text{Liquidity Cost}$$

$$\text{Liquidity Cost}_i = \frac{1}{2} \times \text{spread}_i \times \text{stress multiplier}_i \times |MV_i|$$

Spreads and stress multipliers are illustrative RMP parameters. The output links market risk to trading liquidity by estimating the additional cost of unwinding positions under stress.

<small>

| Asset Class      | Normal Spread | Stress Multiplier | Stressed Spread       |
|------------------|---------------|-------------------|-----------------------|
| Large cap equity | 5bps          | 3x                | 15bps                 |
| Small cap equity | 20bps         | 5x                | 100bps                |
| IG bonds         | 10bps         | 5x                | 50bps                 |
| HY bonds         | 50bps         | 10x               | 500bps               |
| Senior loans     | 100bps        | 20x               | 2000bps              |
| Listed REITs     | 15bps         | 5x                | 75bps                 |
| Direct property  | N/A           | N/A               | 5-8% transaction cost |

</small>

> Note: Liquidity-Adjusted VaR captures trading-liquidity cost. Wider liquidity monitoring, including redemption stress and liquidity buckets, is covered in Section 6.

In [ ]:
# Risk parameters from RMP
liquidity_stress_multiplier = rmp['liquidity_monitoring']['liquidity_stress_multiplier']

In [ ]:
# 2.2 Liquidity-adjusted VaR
# Standard VaR + estimated cost to liquidate under stress

var_1d = var_result['var_hist_pct']
nav = var_result['nav_eur']
lvar_result = rk.liquidity_adjusted_var(var_1d, positions, stress_multiplier=liquidity_stress_multiplier)

phtml.display_lvar(lvar_result, nav, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="07")

## 3. VaR Backtesting and Statistical Diagnostics

VaR estimates are backtested against reconstructed daily portfolio P&L to assess model performance over the testing window. The RMP defines a 250-observation breach-monitoring window and two statistical diagnostics:

- **Kupiec POF Test**: compares observed breaches with the expected breach rate for the selected confidence level.
- **Christoffersen Independence Test**: checks whether breaches are clustered or independent over time.

### Backtesting as Internal Control

Backtesting is used as an internal model-control process. Breach counts and statistical diagnostics help identify when the VaR model may require review, recalibration, or further explanation.

In [ ]:
# Risk parameters from RMP
backtest_lookback = rmp['backtesting']['observation_window']

In [ ]:
# Rolling 1-day VaR backtest over the RMP observation window
from src.risk.var_backtest import compute_var_backtest_rolling, create_backtest_report
import pandas as pd

# Compute rolling VaR backtest
start_date = (pd.Timestamp(VALUATION_DATE) - pd.tseries.offsets.BDay(backtest_lookback)).strftime('%Y-%m-%d')

backtest_df = compute_var_backtest_rolling(
    engine=ENGINE,
    fund_id=FUND_ID,
    start_date=start_date,
    end_date=VALUATION_DATE,
    lookback=backtest_lookback,
)

print(f"✓ Backtest computed: {len(backtest_df)} trading days")
print(f"  Lookback period: {backtest_lookback} days (per date)")
print(f"  Confidence levels: 95%, 97.5%, 99%")

# Create report with Kupiec & Christoffersen tests (95%, 97.5%, 99%)
report = create_backtest_report(backtest_df)

# Display using HTML table
phtml.display_backtest_report(report, window_size=backtest_lookback, valuation_date=VALUATION_DATE, model="Historical", fund_id=FUND_ID, export_id="08")

In [ ]:
from src.ui.backtest_plot import plot_var_backtest

# Extract properly aligned returns and VaR for plotting
# So we use returns[1:] and vars[:-1]
returns_shifted = backtest_df['realised_return'].iloc[1:].values
var_shifted = backtest_df['var_99_pct'].iloc[:-1].values
dates_shifted = backtest_df['date'].iloc[1:].values

# Extract Kupiec and Christoffersen p-values from report 
kupiec_pvalue = report[report['confidence'] == var_confidence*100]['kupiec_p'].values[0] if len(report[report['confidence'] == var_confidence*100]) > 0 else None
chris_pvalue = report[report['confidence'] == var_confidence*100]['christoffersen_p'].values[0] if len(report[report['confidence'] == var_confidence*100]) > 0 else None

fig, ax = plot_var_backtest(
    dates_shifted,
    returns_shifted,
    var_shifted,
    FUND_ID,
    title=f'VaR Backtest — {FUND_ID} (99% confidence, 250-day lookback)',
    kupiec_pvalue=kupiec_pvalue,
    christoffersen_pvalue=chris_pvalue,
    valuation_date=VALUATION_DATE,
    export_id="09"
)

---

## 4. Leverage

Leverage is monitored using both the Gross and Commitment methods.

- **Gross method**: absolute exposures divided by NAV, with derivatives converted to equivalent underlying exposure and no netting.
- **Commitment method**: recognises eligible hedging and netting arrangements, including offsetting positions in the same underlying.

The outputs below compare current leverage against the fund limits and provide a granular breakdown by exposure source.

In [ ]:
from src.risk.leverage_computation import compute_leverage, build_bbg_maps

BORROWINGS_EUR = 0.0

lev = compute_leverage(risk_df, NAV, BBG, FUND_ID, borrowings_eur=BORROWINGS_EUR)

# Extract maps for pre-trade checks
currency_bbg_map, deriv_bbg_map = build_bbg_maps(FUND_ID)

gross_exposure = lev['gross_exposure']
gross_leverage = lev['gross_leverage']
commitment_exposure = lev['commitment_exposure']
commitment_leverage = lev['commitment_leverage']
deriv_notional_commitment = lev['deriv_notional_commitment']

lev_df = lev['risk_df']


In [ ]:
# AIFMD II granular leverage breakdown
from src.risk.leverage_computation import compute_granular_leverage_breakdown

granular = compute_granular_leverage_breakdown(
    risk_df=lev_df,
    nav=NAV,
    borrowings_eur=BORROWINGS_EUR,
)

In [ ]:
phtml.display_granular(granular, NAV, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="10")

---

## 5. Market and Liquidity Stress Testing

Stress testing applies selected market and liquidity shocks to the current portfolio snapshot. The scenarios reflect the main risk factors of a long/short equity fund with fixed income, FX, and derivative exposure.

The project uses a first-order sensitivity approach:

$$\Delta P_i = \text{sensitivity}_i \times \text{shock}_i \times MV_i$$

Scenarios covered:

- **Equity crash**: equity markets down 30%
- **Rate shock**: parallel shift up 200bps
- **Credit widening**: spreads widen 150bps
- **FX stress**: USD and GBP depreciate 15% versus EUR
- **Combined**: simultaneous equity, rate, credit, and FX shock
- **Historical**: 2008 financial crisis, 2020 COVID crash, and 2022 rate shock

> Methodology note: stress P&L uses first-order sensitivities, including equity delta, modified duration for rates and credit, and direct revaluation for FX.

In [ ]:
from src.risk.risk_utils import HISTORICAL_SCENARIOS
phtml.display_historical_scenarios(HISTORICAL_SCENARIOS, fund_id=FUND_ID, export_id="11")

In [ ]:
custom_scenarios = {
    'Equity Crash -30%'      : rk.stress_equity(risk_df, delta_equity=-0.30),
    'Rate Shock +200bps'     : rk.stress_rates(risk_df, delta_y=0.02),
    'Credit Widening +150bps': rk.stress_credit(risk_df, delta_spread=0.015),
    'FX Stress USD/GBP -15%' : rk.stress_fx(risk_df, fx_shocks={'USD': -0.15, 'GBP': -0.15}),
    'Combined'               : rk.stress_combined(risk_df),
}
    
phtml.display_scenarios(risk_df, custom_scenarios, add_historical=True, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="12")

---

## 6. Liquidity Profile

### 6.1 Buckets

The liquidity profile classifies portfolio assets by estimated time to liquidation under normal market conditions. The bucketed view is used to assess whether the fund's liquid assets are aligned with its redemption terms.

<small>

| Bucket | Time to liquidate |
|--------|------------------|
| 1      | 1 day            |
| 2      | 2-7 days         |
| 3      | 8-30 days        |
| 4      | 31-90 days       |
| 5      | 91-365 days      |
| 6      | > 1 year         |

</small>

Days to liquidate are estimated per position as:

$$\text{days}_i = \frac{|MV_i|}{ADV_i \times \text{pct\_adv}}$$

- **ADV** is the average daily volume sourced from the market-data layer.
- **pct_adv** is an RMP parameter defining the maximum share of ADV assumed tradeable per day.
- Cash and money-market instruments are classified as 1 day.
- Instruments with no reliable trading volume are treated as illiquid.

In [ ]:
# Risk parameters from RMP
liquidity_pct_adv = rmp['liquidity_monitoring']['pct_adv']

In [ ]:
# Compute liquidity profile
liq = rk.compute_liquidity_profile(risk_df, pct_adv=liquidity_pct_adv)
risk_df_liq = liq['risk_df_liq']
bucket_full = liq['bucket_full']

phtml.display_buckets(bucket_full, risk_df_liq, NAV, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="13")

In [ ]:
from src.ui.liquidity_plot import plot_liquidity_profile
x = plot_liquidity_profile(bucket_full, FUND_ID, metric='pct_nav_abs', valuation_date=VALUATION_DATE, export_id="14")

---

### 6.2 Redemption Stress

Redemption stress compares assets liquidatable within the fund's notice period against potential redemption requests. A shortfall indicates that the fund may need escalation, liquidity management action, or further portfolio review.

<small>

| Scenario | Redemption | Rationale |
| --- | --- | --- |
| Normal | 10% NAV | Routine investor exit |
| Large | 25% NAV | Large single redemption |
| Stress | 50% NAV | Co-ordinated stress exit |
| Largest investor | Fund-specific | Concentration stress |

</small>

In [ ]:
# Risk parameters from RMP
notice_period_days = rmp['notice_period_days']
notice_period_days

In [ ]:
# Redemption stress scenarios
REDEMPTION_SCENARIOS = [
    (0.10, 'Normal'),
    (0.25, 'Large'),
    (0.50, 'Stress'),
]

phtml.display_redemption_stress(
    FUND_ID,
    notice_period_days,
    REDEMPTION_SCENARIOS,
    NAV,
    risk_df_liq,
    valuation_date=VALUATION_DATE,
    export_id="15",
)

### 6.3 Investor Concentration

Investor concentration is monitored as a fund-level governance indicator:

- Single investor above 20% of NAV is flagged for escalation.
- Top 3 investors above 50% of NAV are flagged as high concentration.

The largest-investor scenario connects the investor register to redemption stress by translating investor concentration into a liquidity demand.

### 6.4 Counterparty Stress

Counterparty stress measures the impact of a prime broker or OTC derivatives counterparty default. The relevant exposure is the net uncollateralised amount after initial margin and variation margin.

> Collateral coverage is simulated in this notebook. The output is used as a governance indicator rather than a live collateral reconciliation.

In [ ]:
# Compute counterparty stress
# Simulated prime brokerage and ISDA OTC derivatives counterparty register
# _cp_hf = rk.load_counterparty(FUND_ID) 
counterparty_stress = rk.compute_counterparty_stress(FUND_ID, ENGINE, NAV)
phtml.display_counterparty_stress(NAV, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="16", **counterparty_stress)

### 6.5 Combined Stress: Market and Liquidity

The combined stress scenario applies an equity market shock and a redemption request at the same time. For this fund, the scenario combines an equity fall of 20% with a redemption request equal to 25% of NAV.

The redemption demand is measured against pre-stress NAV, while liquid assets are reduced by the market shock before assessing coverage.

In [ ]:
# Compute combined market and liquidity stress
phtml.display_combined_stress_mkt_plus_liq(
    risk_df=risk_df,
    risk_df_liq=risk_df_liq,
    nav=NAV,
    notice_days=notice_period_days,
    delta_equity=-0.20,
    redemption_pct=0.25,
    valuation_date=VALUATION_DATE,
    fund_id=FUND_ID,
    export_id="17",
);

#### 6.6 Reconciling Exposure Measures

<small>

| Measure | Numerator | Cash | Shorts | Derivatives |
|---|---|---|---|---|
| NAV | signed market value | included | netted | market value |
| Liquidity buckets | absolute market value | included | absolute value | market value |
| Gross leverage | absolute exposure or notional | excluded | included | full notional |
| Commitment leverage | net exposure | excluded | netted where eligible | delta-adjusted notional |

</small>

These measures answer different questions:

- **NAV** measures portfolio value using signed market values.
- **Liquidity buckets** measure liquidation exposure using absolute market values.
- **Gross leverage** measures total economic exposure before netting.
- **Commitment leverage** recognises eligible netting and hedging.

Gross and commitment leverage are exposure measures, not NAV reconciliation tools.

## 7. P&L Attribution by Risk Factor

P&L attribution explains which risk factors drove daily returns and losses. This section uses sensitivity-based attribution based on current positions and observed market moves.

### Attribution framework

#### A. Total

$$\text{Total P\&L} = \text{Equity P\&L} + \text{Rates P\&L} + \text{FX P\&L} + \text{Residual}$$

#### B. Components

$$\text{Equity P\&L} = \sum_i \beta_i \times r_{market} \times MV_i$$

$$\text{Rates P\&L} = \sum_i -D_i \times \Delta y \times MV_i$$

$$\text{FX P\&L} = \sum_i \text{notional}_{foreign,i} \times r_{FX,i}$$

where $\beta_i$ is the 1-year rolling beta to the benchmark, $D_i$ is modified duration, and $r_{FX,i}$ is the daily FX return versus EUR.

#### C. Residual

$$\text{Residual} = \text{P\&L}_{actual} - (\text{Equity} + \text{Rates} + \text{FX})$$

A large or persistent residual may point to missing factors, model limitations, or data issues. It is shown separately rather than absorbed into the explained components.

$$\%\text{ explained} = |\text{explained}| / |\text{actual}|$$

Target explained P&L: $\geq 80\%$.

In [ ]:
from src.risk.pnl_attribution import compute_daily_attribution
from src.ui.attribution_plot import plot_attribution_cumsum

# Compute attribution
attr_result = compute_daily_attribution(
    engine=ENGINE,
    fund_id=FUND_ID,
    bbg=BBG,
    valuation_date=VALUATION_DATE,
    residual_threshold_pct=0.20,
)

attr = attr_result['attr']
flagged = attr_result['flagged']
attr_cumsum = attr_result['attr_cumsum']

# Plot
plot_attribution_cumsum(attr_cumsum, FUND_ID, valuation_date=VALUATION_DATE, export_id="18")

# Model quality report
import src.print_utils as prt
prt.print_attribution(attr, flagged)

**Methodology and limitations**

The attribution uses actual daily positions and market moves. Regression-based attribution is not used because it gives average historical loadings rather than current portfolio exposures.

Benchmark: SPY (S&P 500), used as the equity market factor for this USD-heavy long/short equity portfolio.

**Limitations:**

- Single-factor equity model, with no sector, style, or stock-level decomposition.
- Rate attribution uses a simulated parallel shift, not key-rate DV01.
- FX attribution covers EUR/USD only.
- Position composition is static in this simulation.

## 8. Pre-Trade Compliance Check

The pre-trade check assesses the impact of a proposed transaction before execution. It builds a pro-forma portfolio and compares post-trade risk metrics against the fund's RMP limits.

The workflow:

1. Loads the current enriched portfolio using `get_risk_ready_df`.
2. Applies the proposed transaction to create a pro-forma position set.
3. Runs leverage, concentration, counterparty, short-selling, and liquidity checks.

### Checks performed

<small>

| # | Check | Limit | Basis |
|---|---|---|---|
| 1 | Gross leverage | 300% NAV | EU 231/2013 Art. 7; Recitals 13-14 |
| 2 | Commitment leverage | 200% NAV | EU 231/2013 Art. 8 |
| 3 | Single-issuer concentration | 25% NAV | Internal RMP |
| 4 | Sector concentration | 30% NAV | Internal RMP |
| 5 | Counterparty exposure | 10% / 5% NAV | EU 231/2013 Art. 43 |
| 6 | Net short position | 0.2% NAV reportable | EU 236/2012 Art. 5 |
| 7 | Liquidity days to liquidate | Fund-specific | RMP liquidity parameters |

</small>

### Borrowing treatment

Borrowings are included in both gross and commitment exposure at absolute value. The `pct_financed` field controls whether the proposed trade is cash-funded or financed through borrowing.

- `pct_financed = 0.0`: cash-funded trade, with cash reduced by the full notional.
- `pct_financed = 1.0`: prime-broker financed trade, with a borrowing row added to the pro-forma portfolio.

### RMP link

The RMP provides the leverage and concentration limits used to determine whether the proposed trade passes or fails.

### Pre-existing breaches

Single-issuer and net short-position checks only flag a breach when the proposed trade worsens the existing position.


### 8.1 Scenario 1 - Clean Buy

A new investment-grade Deutsche Bank bond is added to the portfolio. The EUR 3M notional represents approximately 3.2% of NAV.

Expected result: **PASS**.

The trade remains below the gross leverage limit and below the single-issuer concentration limit.

In [ ]:
from src.risk.leverage_computation import build_bbg_maps
counterparties_df = rk.load_counterparty(FUND_ID)
currency_bbg_map, deriv_bbg_map = build_bbg_maps(FUND_ID)

PTC_CTX = dict(
    engine=ENGINE,
    fund_id=FUND_ID,
    date=VALUATION_DATE,
    counterparties_df=counterparties_df,
    bbg=BBG,
    deriv_bbg_map=deriv_bbg_map,
    currency_bbg_map=currency_bbg_map,
    rmp=rmp, # Pass the entire RMP for access to all parameters in pre-trade checks
)

In [ ]:
trade_pass = dict(
    isin='XS1234567890',
    direction='BUY',
    quantity=3_000_000,
    price_eur=100.50,
    counterparty='Deutsche Bank',
    underlying_risk='Deutsche Bank AG (DBK)',
    asset_class='Bonds',
    sub_asset_class='Corporate IG',
    dur_adj_mid=5.2,
    currency='EUR',
    adv_eur=2_500_000,
    issuer='Deutsche Bank AG',
    sector='Financials',
    pct_financed=0.0,
)

result_pass = rk.pre_trade_check(trade_pass, **PTC_CTX)
phtml.display_ptc(result_pass, test_number=1, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="19")

### 8.2 Scenario 2 - Gross and Commitment Leverage Breach

A EUR 150M Goldman Sachs equity position is added to the portfolio. The trade is fully prime-broker financed and increases total exposure above the 300% gross leverage limit.

Expected result: **FAIL**.

The scenario is designed to trigger several breaches at once: gross leverage, commitment leverage, and single-issuer concentration.

In [ ]:
trade_leverage = dict(
    isin='US0200021014',
    direction='BUY',
    quantity=1_500_000,
    price_eur=100.00,
    counterparty='Goldman Sachs',
    underlying_risk='Goldman Sachs Group Inc (GS)',
    asset_class='Equity',
    sub_asset_class='Large Cap',
    dur_adj_mid=0.0,
    currency='USD',
    adv_eur=50_000_000,
    issuer='Goldman Sachs Group Inc',
    sector='Financials',
    pct_financed=1.0,
)

result_leverage = rk.pre_trade_check(trade_leverage, **PTC_CTX)
phtml.display_ptc(result_leverage, test_number=2, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="20")

### 8.3 Scenario 3 - Single-Issuer Concentration Breach

A new Alphabet position of EUR 25.5M is added to the portfolio. On a NAV of approximately EUR 94.5M, the position represents about 27.0% of NAV, above the 25% single-issuer concentration limit.

Expected result: **FAIL**.

The trade is sized to breach concentration while remaining within the gross leverage limit.

In [ ]:
trade_conc = dict(
    isin='US0255371017',
    direction='BUY',
    quantity=600_000,
    price_eur=42.50,
    counterparty='Barclays',
    underlying_risk='Alphabet Inc (GOOGL)',
    asset_class='Equity',
    sub_asset_class='Large Cap',
    dur_adj_mid=0.0,
    currency='USD',
    adv_eur=80_000_000,
    issuer='Alphabet Inc',
    sector='Technology',
    pct_financed=0.0,
)

result_conc = rk.pre_trade_check(trade_conc, **PTC_CTX)
phtml.display_ptc(result_conc, test_number=3, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="21")

---

## 9. Sustainability Risk Indicators

Sustainability risk indicators are monitored as part of fund governance. The fund is classified as SFDR Article 6 and does not have an explicit sustainability objective.

Portfolio-level ESG indicators are calculated using NAV-weighted exposures.

Metrics monitored:

- Weighted average ESG score, including composite, E, S, and G scores.
- Percentage of NAV in instruments with ESG score below the RMP threshold.
- Percentage of NAV with an active controversy flag.
- Weighted average carbon intensity, measured in tCO2 / EURm revenue.

ESG scores for liquid instruments are fetched from the market-data layer. Instruments without a Bloomberg ticker use fund administrator ESG data embedded in the position file.

> Scale note: ESG scores use a 0-100 scale, where 100 is best. The RMP threshold used in this notebook is 30.

> Derivatives look-through: options use delta-adjusted notional as the ESG exposure weight, futures use full notional, and FX forwards are assigned no issuer ESG exposure.
>
> For options:
>
> $$ESG\_exposure_i = |\delta_i| \times Q_i \times \text{contract size} \times P_{underlying} \times FX$$


In [ ]:
import src.risk.esg_utils as esg_u

esg_df  = esg_u.build_esg_df(risk_df, BBG, ENGINE, FUND_ID, VALUATION_DATE)
esg_u.display_esg_assets(esg_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="22")

In [ ]:
esg_u.display_esg_summary(esg_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="23")

In [ ]:
esg_u.plot_esg_profile(esg_df, FUND_ID, plot_title='06. ESG profile - HF', valuation_date=VALUATION_DATE, export_id="24")

---

## 10. Annex IV Report

Selected outputs from the risk snapshot are mapped to Annex IV-style reporting fields, including fund identity, strategy, risk profile, leverage, liquidity profile, and fund-level risk indicators.

The leverage section includes both Gross and Commitment methods, together with a granular breakdown by exposure source where available.

**Regulatory basis:** Delegated Regulation (EU) 231/2013 Article 110 and Annex IV reporting template.


In [ ]:
# Build Annex IV data
from src.ui.annex_iv_display import annex_iv_section
from src.reporting.annex_iv import build_annex_iv, export_annex_iv_excel

ANNEX_IV_QUARTER = '2026-03-31'
annex_iv = build_annex_iv(ENGINE, FUND_ID, quarter=ANNEX_IV_QUARTER)

# Display Annex IV sections
annex_iv_section(annex_iv, 'identification', fund_id=FUND_ID, export_id="25");
annex_iv_section(annex_iv, 'breakdown', fund_id=FUND_ID, export_id="26");
annex_iv_section(annex_iv, 'risk_measures', fund_id=FUND_ID, export_id="27");
annex_iv_section(annex_iv, 'leverage_detail', fund_id=FUND_ID, export_id="28");
annex_iv_section(annex_iv, 'liquidity_buckets', fund_id=FUND_ID, export_id="29");
annex_iv_section(annex_iv, 'liquidity_terms', fund_id=FUND_ID, export_id="30");

# Export to Excel
path = export_annex_iv_excel(ENGINE, quarter=ANNEX_IV_QUARTER, output_dir='../../data')
print(f"Annex IV workbook written: {path}")